# OpenAI 适配器 (旧)

**请确保 OpenAI 库版本低于 1.0.0；否则，请参考新版文档 [OpenAI 适配器](/docs/integrations/adapters/openai/)。**

许多人从 OpenAI 入门，但希望探索其他模型。LangChain 与众多模型提供商的集成使这一过程变得轻松。虽然 LangChain 拥有自己的消息和模型 API，但我们也通过公开一个适配器来使探索其他模型变得尽可能容易，该适配器可以将 LangChain 模型适配到 OpenAI API。

目前这仅处理输出，而不返回其他信息（令牌计数、停止原因等）。

In [ ]:
import openai
from langchain_community.adapters import openai as lc_openai

## ChatCompletion.create

In [29]:
messages = [{"role": "user", "content": "hi"}]

原始 OpenAI 调用

In [15]:
result = openai.ChatCompletion.create(
    messages=messages, model="gpt-3.5-turbo", temperature=0
)
result["choices"][0]["message"].to_dict_recursive()

{'role': 'assistant', 'content': 'Hello! How can I assist you today?'}

LangChain OpenAI 包装器调用

LangChain 提供了一个 OpenAI 包装器，用于与 OpenAI 的各种模型进行交互。此包装器简化了与这些模型的 API 调用，并提供了与 LangChain 框架集成的能力。

以下是如何使用 LangChain 的 OpenAI 包装器的示例：

```python
from langchain_openai import OpenAI

# 初始化 OpenAI 包装器
# 您可以指定模型名称，例如 "gpt-3.5-turbo-instruct" 或 "text-davinci-003"
llm = OpenAI(model_name="gpt-3.5-turbo-instruct")

# 调用模型进行文本生成
prompt = "请写一首关于人工智能的短诗。"
response = llm.invoke(prompt)

print(response)
```

**关键参数和功能：**

*   **`model_name`**: 指定要使用的 OpenAI 模型。常见的模型包括 `gpt-3.5-turbo-instruct`、`text-davinci-003` 等。
*   **`temperature`**: 控制生成文本的随机性。较高的值（例如 0.8）会产生更多样化和创造性的输出，而较低的值（例如 0.2）会产生更集中和确定的输出。
*   **`max_tokens`**: 生成文本的最大长度（以 token 为单位）。
*   **`top_p`**: 另一种控制随机性的方法，称为“核心采样”。模型会考虑累积概率达到 `top_p` 的 token。
*   **`frequency_penalty`**: 负值会鼓励模型重复使用与先前生成的 token 相同的 token。
*   **`presence_penalty`**: 负值会鼓励模型谈论新的话题。

**与 LangChain 的集成：**

OpenAI 包装器可以轻松地与 LangChain 的其他组件（如链（Chains）、代理（Agents）和记忆（Memory））集成。这使得构建复杂的 LLM 应用程序成为可能。

例如，您可以将 OpenAI 包装器与 `LLMChain` 结合使用：

```python
from langchain_openai import OpenAI
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate

# 初始化 OpenAI 包装器
llm = OpenAI(model_name="gpt-3.5-turbo-instruct", temperature=0.7)

# 定义一个提示模板
prompt_template = PromptTemplate(
    input_variables=["topic"],
    template="请写一篇关于 {topic} 的简短文章。"
)

# 创建一个 LLMChain
chain = LLMChain(llm=llm, prompt=prompt_template)

# 运行链
topic = "机器学习"
response = chain.invoke({"topic": topic})

print(response['text'])
```

**注意事项：**

*   在使用 OpenAI 包装器之前，请确保您已安装 `langchain-openai` 包 (`pip install langchain-openai`)。
*   您需要设置 OpenAI API 密钥。通常可以通过环境变量 `OPENAI_API_KEY` 来设置。
*   根据您使用的模型和 API 调用量，可能会产生费用。

In [17]:
lc_result = lc_openai.ChatCompletion.create(
    messages=messages, model="gpt-3.5-turbo", temperature=0
)
lc_result["choices"][0]["message"]

{'role': 'assistant', 'content': 'Hello! How can I assist you today?'}

更换模型提供商

You can swap out the model provider for any component that uses one.
您可以为任何使用模型提供商的组件更换模型提供商。

For example, if you want to use OpenAI's models instead of the default ones, you can do so by creating a new `OpenAI` instance and passing it to the component.
例如，如果您想使用 OpenAI 的模型而不是默认模型，可以通过创建一个新的 `OpenAI` 实例并将其传递给组件来实现。

```jsx
import { ChatCompletion } from "@langchain/openai";

const ollama = new OpenAI(); // Default to OpenAI
const openAI = new OpenAI({ apiKey: "YOUR_API_KEY" });

const chat = new ChatCompletion({
  model: "gpt-3.5-turbo",
  temperature: 0.7,
  // You can pass any OpenAI constructor options here
  // 您可以在此处传递任何 OpenAI 构造函数选项
});

const chat2 = new ChatCompletion({
  model: "gpt-4",
  temperature: 0.7,
  // You can pass any OpenAI constructor options here
  // 您可以在此处传递任何 OpenAI 构造函数选项
});

// You can also swap out the model provider for a specific component
// 您也可以为特定组件更换模型提供商
const chat3 = new ChatCompletion({
  model: "gpt-3.5-turbo",
  temperature: 0.7,
  // You can pass any OpenAI constructor options here
  // 您可以在此处传递任何 OpenAI 构造函数选项
});

In [19]:
lc_result = lc_openai.ChatCompletion.create(
    messages=messages, model="claude-2", temperature=0, provider="ChatAnthropic"
)
lc_result["choices"][0]["message"]

{'role': 'assistant', 'content': ' Hello!'}

## ChatCompletion.stream

原始 OpenAI 调用

In [24]:
for c in openai.ChatCompletion.create(
    messages=messages, model="gpt-3.5-turbo", temperature=0, stream=True
):
    print(c["choices"][0]["delta"].to_dict_recursive())

{'role': 'assistant', 'content': ''}
{'content': 'Hello'}
{'content': '!'}
{'content': ' How'}
{'content': ' can'}
{'content': ' I'}
{'content': ' assist'}
{'content': ' you'}
{'content': ' today'}
{'content': '?'}
{}


LangChain OpenAI 包装器调用

LangChain 提供了一个 OpenAI 包装器，用于与 OpenAI API 进行交互。此包装器简化了与各种 OpenAI 模型（如 GPT-3.5、GPT-4 等）的集成。

以下是如何使用 LangChain 的 OpenAI 包装器的示例：

```python
from langchain_openai import OpenAI

# 初始化 OpenAI 包装器
# 您可以指定模型名称、温度等参数
llm = OpenAI(model_name="gpt-3.5-turbo-instruct", temperature=0.7)

# 调用包装器以生成文本
prompt = "写一个关于人工智能的短故事。"
response = llm.invoke(prompt)

print(response)
```

**关键参数：**

*   **`model_name`**: 指定要使用的 OpenAI 模型。常见的选项包括 `"gpt-3.5-turbo-instruct"`、`"gpt-4"` 等。
*   **`temperature`**: 控制生成文本的随机性。较高的值（例如 0.8）会产生更多样化的输出，而较低的值（例如 0.2）会产生更集中、更确定的输出。
*   **`max_tokens`**: 生成响应的最大 token 数。
*   **`top_p`**: 使用核采样（nucleus sampling）进行采样，其中模型考虑了累积概率为 `top_p` 的 token。
*   **`frequency_penalty`**: 负的频率惩罚。正值会根据它们在文本中出现的频率来惩罚新 token，从而鼓励模型谈论更多新主题。
*   **`presence_penalty`**: 负的存在惩罚。正值会根据它们是否已出现在文本中来惩罚新 token，从而鼓励模型谈论更多新主题。

**高级用法：**

*   **流式输出**: 您可以配置包装器以流式传输响应，从而在生成时接收 token。
*   **自定义请求参数**: LangChain 允许您将 OpenAI API 支持的任何其他参数传递给包装器。
*   **与链集成**: OpenAI 包装器可以轻松地与 LangChain 的其他组件（如提示模板、输出解析器和代理）集成，以构建复杂的 LLM 应用程序。

**示例：流式输出**

```python
from langchain_openai import OpenAI

llm = OpenAI(model_name="gpt-3.5-turbo-instruct", temperature=0.7, streaming=True)

prompt = "写一个关于人工智能的短故事。"
for chunk in llm.stream(prompt):
    print(chunk, end="", flush=True)
```

通过使用 LangChain 的 OpenAI 包装器，您可以有效地利用 OpenAI 的强大功能来构建各种自然语言处理应用程序。

In [30]:
for c in lc_openai.ChatCompletion.create(
    messages=messages, model="gpt-3.5-turbo", temperature=0, stream=True
):
    print(c["choices"][0]["delta"])

{'role': 'assistant', 'content': ''}
{'content': 'Hello'}
{'content': '!'}
{'content': ' How'}
{'content': ' can'}
{'content': ' I'}
{'content': ' assist'}
{'content': ' you'}
{'content': ' today'}
{'content': '?'}
{}


更换模型提供商

You can swap out the model provider for a given component.
您可以为给定组件更换模型提供商。

For example, if you want to use `ChatOpenAI` instead of `ChatAnthropic`, you can do the following:
例如，如果您想使用 `ChatOpenAI` 而不是 `ChatAnthropic`，可以这样做：

```jsx
import { ChatOpenAI } from "@langchain/openai";
import { ChatMessage } from "@langchain/core/messages";

const chat = new ChatOpenAI({ modelName: "gpt-3.5-turbo" });
const messages = [
  new ChatMessage({ content: "Hello, world!", role: "human" }),
];

const response = await chat.invoke({ messages });

console.log(response);
```

This example shows how to swap out the model provider for a given component.
此示例展示了如何为给定组件更换模型提供商。

You can also swap out the model provider for a given component by using the `withProvider` method.
您还可以通过使用 `withProvider` 方法为给定组件更换模型提供商。

```jsx
import { ChatAnthropic } from "@langchain/anthropic";
import { ChatOpenAI } from "@langchain/openai";
import { ChatMessage } from "@langchain/core/messages";

const chat = new ChatAnthropic({ modelName: "claude-2" });
const messages = [
  new ChatMessage({ content: "Hello, world!", role: "human" }),
];

const response = await chat.withProvider(new ChatOpenAI({ modelName: "gpt-3.5-turbo" })).invoke({ messages });

console.log(response);
```

This example shows how to swap out the model provider for a given component by using the `withProvider` method.
此示例展示了如何通过使用 `withProvider` 方法为给定组件更换模型提供商。

In [31]:
for c in lc_openai.ChatCompletion.create(
    messages=messages,
    model="claude-2",
    temperature=0,
    stream=True,
    provider="ChatAnthropic",
):
    print(c["choices"][0]["delta"])

{'role': 'assistant', 'content': ' Hello'}
{'content': '!'}
{}
